## Objetivo
Analizar patrones asociados al abandono de clientes (churn) e identificar
variables demográficas, de uso y facturación relacionadas con una mayor
probabilidad de abandono.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
sns.set(style="whitegrid")

In [ ]:
# Carga de datos limpios
df = pd.read_excel("../data/processed/customers_churn_clean.xlsx")

print(f"Filas: {df.shape[0]} | Columnas: {df.shape[1]}")
df.head()

El dataset contiene información demográfica, características del servicio,
uso mensual, facturación y el estado de churn de cada cliente.

In [ ]:
df.info()
df.describe()

## Análisis de la Variable Objetivo: Churn
Se analiza la distribución general de clientes que abandonan el servicio
frente a los que permanecen activos.

In [ ]:
churn_rate = df["churn"].value_counts(normalize=True) * 100
churn_rate

In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(
    data=df,
    x="churn",
    hue="churn",
    palette="Set2",
    legend=False
)
plt.title("Distribución de Churn")
plt.ylabel("Número de clientes")
plt.xlabel("Churn")
plt.show()
plt.close()

**Insight:** El churn representa una proporción relevante del total de clientes,
lo que indica una oportunidad clara para implementar estrategias de retención.

## Análisis Univariado
Se exploran las distribuciones individuales de las principales variables
numéricas y categóricas.

In [ ]:
numeric_cols = [
    "age",
    "tenure_months",
    "monthly_usage_gb",
    "monthly_charges",
    "total_charges",
    "support_calls"
]

for col in numeric_cols:
    plt.figure(figsize=(6,4))
    sns.histplot(df[col], kde=True)
    plt.title(f"Distribución de {col}")
    plt.show()
    plt.close()

In [ ]:
categorical_cols = [
    "gender",
    "contract_type",
    "internet_service",
    "senior_citizen"
]

for col in categorical_cols:
    plt.figure(figsize=(6,4))
    sns.countplot(data=df, x=col)
    plt.title(f"Distribución de {col}")
    plt.xticks(rotation=30)
    plt.show()
    plt.close()

Se observan diferencias claras en la composición de clientes según tipo de
contrato y servicio de internet, lo cual será relevante al analizar el churn.

## Análisis Bivariado: Variables vs Churn
Se comparan las características de clientes activos y churned para identificar
factores asociados al abandono.

In [ ]:
for col in numeric_cols:
    plt.figure(figsize=(6,4))
    sns.boxplot(data=df, x="churn", y=col)
    plt.title(f"{col} vs Churn")
    plt.show()
    plt.close()

In [ ]:
for col in categorical_cols:
    churn_segment = (
        df.groupby(col)["churn"]
        .value_counts(normalize=True)
        .rename("rate")
        .reset_index()
    )

    churn_segment = churn_segment[churn_segment["churn"] == "Yes"]

    plt.figure(figsize=(6,4))
    sns.barplot(data=churn_segment, x=col, y="rate")
    plt.title(f"Tasa de Churn por {col}")
    plt.ylabel("Churn Rate")
    plt.xticks(rotation=30)
    plt.show()
    plt.close()

**Insight:** Los clientes con contrato mensual y menor antigüedad presentan
tasas de churn significativamente más altas.

## Segmentación por Antigüedad
Se analiza el churn según grupos de tenure para identificar etapas críticas
del ciclo de vida del cliente.

In [ ]:
plt.figure(figsize=(6,4))
sns.barplot(
    data=df,
    x="tenure_group",
    y=(df["churn"] == "Yes").astype(int)
)
plt.title("Tasa de Churn por Grupo de Antigüedad")
plt.ylabel("Churn Rate")
plt.xlabel("Tenure Group")
plt.show()
plt.close()

El churn es más elevado durante los primeros meses, lo que sugiere que las
acciones de retención tempranas son críticas.

## Correlación entre Variables Numéricas
Se analizan relaciones lineales entre variables numéricas para detectar
posibles asociaciones relevantes.

In [ ]:
corr = df[numeric_cols].corr()

plt.figure(figsize=(8,6))
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Matriz de Correlación")
plt.show()
plt.close()

## Conclusiones Finales

- El churn se concentra en clientes con contratos mensuales.
- La baja antigüedad es uno de los principales factores asociados al abandono.
- Un mayor número de llamadas a soporte está relacionado con mayor churn.

**Próximos pasos:** profundizar en estrategias de retención y, opcionalmente,
construir un modelo simple de predicción de churn.